# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [35]:
from pyspark.sql.functions import col, count, countDistinct, coalesce, lit

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [8]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Joining Citiations and Parents to extract relation between CITED, CITING and CITED_STATE

In [17]:
cited = citations\
        .join(patents, citations.CITED==patents.PATENT, "left")\
        .select(citations.CITED, citations.CITING, patents.POSTATE.alias("CITED_STATE"))\
        .cache()
cited.show(10)

+-----+-------+-----------+
|CITED| CITING|CITED_STATE|
+-----+-------+-----------+
| 2366|4192521|       NULL|
| 2366|4253355|       NULL|
| 2366|4305315|       NULL|
| 5156|5580635|       NULL|
| 5518|4976561|       NULL|
+-----+-------+-----------+
only showing top 5 rows



## Now joining the resultant cited with patents to obtain the intermediate result DataFrame so that we can proceed with filtering and counts

In [19]:
cited_citing = cited.alias("b")\
                .join(patents.alias("p"), col("b.CITING")==col("p.PATENT"),"left")\
                .select(col("b.CITED"),col("b.CITED_STATE"),col("b.CITING"),col("p.POSTATE").alias("CITING_STATE"))\
                .cache()
cited_citing.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|1540798|       NULL|3858258|          CA|
|1331793|       NULL|3858258|          CA|
|3638586|         CA|3858527|        NULL|
| 924225|       NULL|3858527|        NULL|
|2444326|       NULL|3858527|        NULL|
|3699902|         OH|3858527|        NULL|
|2967080|       NULL|3858527|        NULL|
|3602157|         TX|3858527|        NULL|
|2705120|       NULL|3858527|        NULL|
| 957631|       NULL|3858560|          IN|
+-------+-----------+-------+------------+
only showing top 10 rows



In the above steps, we have taken left join to preserve the cited patents which might be absent in the patents DB.
They can be identified by their NULL state values.
Also caching these Dataframes to avoid repetitive operations

## Filtering out NULL values and non-matching pairs to prepare for count calculation

In [50]:
filtered_cites = cited_citing.filter(
    (col("CITED_STATE").isNotNull()) &
    (col("CITING_STATE").isNotNull()) &
    (col("CITED_STATE") == col("CITING_STATE"))
)
filtered_cites.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|3368197|         MI|3859627|          MI|
|3722929|         CA|3860191|          CA|
|3172282|         AZ|3861180|          AZ|
|3802510|         MA|3861473|          MA|
|3791450|         MA|3861473|          MA|
|3118651|         MI|3862577|          MI|
|3099569|         PA|3862844|          PA|
|3769543|         NY|3863090|          NY|
|3467396|         MI|3863935|          MI|
|3167490|         NY|3864160|          NY|
+-------+-----------+-------+------------+
only showing top 10 rows



## Counting same states for the citing patents

In [33]:
#Counting states
count_cites = filtered_cites.groupBy("CITING")\
                .agg(count("*").alias("SAME_STATE"))\
                .orderBy(col("SAME_STATE").desc(),col("CITING").asc())
count_cites.show(10)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|5959466|       125|
|5983822|       103|
|6008204|       100|
|5952345|        98|
|5958954|        96|
|5998655|        96|
|5936426|        94|
|5739256|        90|
|5913855|        90|
|5925042|        90|
+-------+----------+
only showing top 10 rows



# Generating the final result

Joining the count table with patents to present the final result about patent info for each citing patent.


In [36]:
result = (
    patents
    .join(count_cites, patents.PATENT == count_cites.CITING,"left")
    .withColumn(
        "SAME_STATE", coalesce(count_cites.SAME_STATE,lit(0)))
    .drop("CITING")
)   

In [48]:
top10=result.orderBy(col("SAME_STATE").desc())
top10.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 

#### RESULT: Top 10 counts being displayed above representing the largest number of citations a patents has from its same state.